# 🎓 Classroom AI — ArcFace Embedding Generator

Notebook này giúp tạo face embeddings cho hệ thống điểm danh AI.

## Workflow:
1. **Upload** file `face_photos_*.zip` từ Dashboard
2. **Chạy** InsightFace ArcFace model (GPU miễn phí)
3. **Download** file `deep_embeddings.pkl` về máy
4. **Import** vào Dashboard qua trang Học sinh → Import Embeddings

> ⚡ Sử dụng GPU T4 miễn phí trên Colab → nhanh hơn 10-50x so với CPU

In [ ]:
# ============================================================
# BƯỚC 1: Cài đặt thư viện
# ============================================================
!pip install insightface onnxruntime-gpu opencv-python-headless -q
print('✅ Cài đặt xong!')

In [ ]:
# ============================================================
# BƯỚC 2: Upload file ZIP ảnh khuôn mặt
# ============================================================
from google.colab import files
import zipfile
import os

print('📸 Chọn file face_photos_*.zip từ Dashboard...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'📦 Đã nhận: {zip_name} ({len(uploaded[zip_name])/1024:.1f} KB)')

# Giải nén
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('.')

# Kiểm tra cấu trúc
photo_dir = 'face_photos'
if not os.path.exists(photo_dir):
    # Thử tìm trong thư mục con
    for d in os.listdir('.'):
        if os.path.isdir(d) and any(os.path.isdir(os.path.join(d, s)) for s in os.listdir(d)):
            photo_dir = d
            break

students = [d for d in sorted(os.listdir(photo_dir)) if os.path.isdir(os.path.join(photo_dir, d))]
print(f'\n✅ Tìm thấy {len(students)} học sinh:')
for s in students:
    photos = [f for f in os.listdir(os.path.join(photo_dir, s)) if f.startswith('sample_')]
    print(f'   {s}: {len(photos)} ảnh')

In [ ]:
# ============================================================
# BƯỚC 3: Khởi tạo InsightFace ArcFace model
# ============================================================
import insightface
from insightface.app import FaceAnalysis
import cv2
import numpy as np
import json

# Load model buffalo_l (ArcFace) — auto-download nếu chưa có
print('🧠 Đang load InsightFace ArcFace model...')
app = FaceAnalysis(
    name='buffalo_l',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
app.prepare(ctx_id=0, det_size=(640, 640))
print('✅ Model ready! (buffalo_l / ArcFace / 512-dim)')

In [ ]:
# ============================================================
# BƯỚC 4: Generate embeddings cho từng học sinh
# ============================================================
from datetime import datetime

embeddings_db = {}
errors = []

for student_id in students:
    student_dir = os.path.join(photo_dir, student_id)
    
    # Đọc metadata nếu có
    meta_path = os.path.join(student_dir, 'metadata.json')
    if os.path.exists(meta_path):
        with open(meta_path, 'r', encoding='utf-8') as f:
            meta = json.load(f)
        name = meta.get('name', student_id)
        class_name = meta.get('class_name', '')
    else:
        name = student_id
        class_name = ''
    
    # Thu thập embeddings từ tất cả ảnh
    student_embeddings = []
    photo_files = sorted([
        f for f in os.listdir(student_dir)
        if f.startswith('sample_') and f.endswith(('.png', '.jpg', '.jpeg'))
    ])
    
    for photo_file in photo_files:
        img_path = os.path.join(student_dir, photo_file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        
        # Detect faces
        faces = app.get(img)
        if not faces:
            # Nếu không detect được, thử resize lên
            h, w = img.shape[:2]
            if h < 112 or w < 112:
                scale = max(112/h, 112/w) * 1.5
                img = cv2.resize(img, (int(w*scale), int(h*scale)))
                faces = app.get(img)
        
        if faces:
            # Lấy khuôn mặt lớn nhất
            face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
            emb = face.embedding
            # L2 normalize
            emb = emb / np.linalg.norm(emb)
            student_embeddings.append(emb.astype(np.float32))
        else:
            errors.append(f'{student_id}/{photo_file}: Không detect được khuôn mặt')
    
    if student_embeddings:
        # Tính centroid (trung bình)
        centroid = np.mean(student_embeddings, axis=0)
        centroid = centroid / np.linalg.norm(centroid)
        
        embeddings_db[student_id] = {
            'name': name,
            'student_id': student_id,
            'class_name': class_name,
            'embeddings': student_embeddings,
            'centroid': centroid.astype(np.float32),
            'enrolled_at': datetime.now().isoformat(),
            'source': 'google_colab_insightface',
            'photo_count': len(student_embeddings),
        }
        print(f'  ✅ {student_id} ({name}): {len(student_embeddings)}/{len(photo_files)} embeddings')
    else:
        print(f'  ❌ {student_id} ({name}): Không tạo được embedding nào')

print(f'\n📊 Kết quả: {len(embeddings_db)}/{len(students)} học sinh có embeddings')
if errors:
    print(f'⚠️ {len(errors)} lỗi:')
    for e in errors[:10]:
        print(f'   {e}')

In [ ]:
# ============================================================
# BƯỚC 5: Lưu và download file embeddings
# ============================================================
import pickle

output = {
    'version': '2.0',
    'model': 'insightface_buffalo_l_arcface',
    'embedding_dim': 512,
    'threshold_recommended': 0.45,
    'created_at': datetime.now().isoformat(),
    'created_by': 'Google Colab + InsightFace',
    'total_students': len(embeddings_db),
    'total_embeddings': sum(len(s['embeddings']) for s in embeddings_db.values()),
    'students': embeddings_db,
}

pkl_path = 'deep_embeddings.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(output, f, protocol=pickle.HIGHEST_PROTOCOL)

size_kb = os.path.getsize(pkl_path) / 1024
print(f'\n💾 Đã lưu: {pkl_path} ({size_kb:.1f} KB)')
print(f'   {output["total_students"]} học sinh, {output["total_embeddings"]} embeddings')
print(f'   Model: {output["model"]}')
print(f'   Threshold: {output["threshold_recommended"]}')

# Auto-download
print('\n📥 Downloading...')
files.download(pkl_path)
print('\n✅ XONG! Upload file này vào Dashboard → Học sinh → Import Embeddings')

---

## 📋 Hướng dẫn tiếp theo

1. Mở Dashboard → Trang **👤 Học sinh**
2. Click nút **📥 Import Embeddings**
3. Chọn file `deep_embeddings.pkl` vừa download
4. Hệ thống sẽ tự động chuyển sang ArcFace engine
5. Bắt đầu buổi học → Điểm danh tự động với độ chính xác cao!

### ⚙️ Tùy chỉnh threshold

- **0.40**: Nhạy hơn (dễ nhận diện nhưng có thể nhầm)
- **0.45**: Mặc định (cân bằng)
- **0.55**: Chặt hơn (ít nhầm nhưng có thể bỏ sót)

### 💡 Mẹo tăng accuracy

- Chụp 5-10 ảnh/HS với nhiều góc: chính diện, nghiêng trái/phải, hơi ngẩng/cúi
- Đảm bảo ánh sáng tốt, khuôn mặt rõ ràng
- Chụp trong điều kiện tương tự lớp học (ánh sáng, khoảng cách)